# Frozen full-AIT valence directions on SST, IMDb, and DynaSent

This notebook loads the **AIT-validation-selected layer and saved direction checkpoint** for mean difference and one-dimensional DAS from full-AIT run `2026-09-23_09-10_CDT`. It evaluates those frozen choices on SST, IMDb, DynaSent R1, and DynaSent R2 for GPT-2 Small and Qwen3-0.6B Base. Logistic regression is intentionally excluded because it performed poorly in the full-AIT experiment.

No direction is refit and no layer, checkpoint, method, or intervention strength is selected from the four transfer datasets. Mean difference retains final-token patching; DAS retains the all-token layer-selection/evaluation protocol used by the full-AIT experiment.

## Before running

1. Use a Colab GPU runtime.
2. Confirm the completed source run exists under `MyDrive/sentiment-geometry/full-ait-last-token-directions/runs`.
3. Provide a Hugging Face token with read access to the private SST, IMDb, and DynaSent repositories when prompted. The token is removed after evaluation.
4. Leave the pinned model and dataset revisions unchanged for a reportable run.
5. For a fresh run, leave `RESUME_RUN_ID = None` and evaluate both models. To resume an interrupted run, set its ID, place completed models in `REUSE_COMPLETED_MODEL_NAMES`, and remove them from `EVALUATION_MODEL_NAMES`.

In [ ]:
# --------------------------- User settings ---------------------------
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional commit or tag. Existing checkouts must match it.
CONFIG_RELATIVE_PATH = "configs/ait_valence_transfer_evaluation.yaml"

DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
SOURCE_RUN_ID = "2026-09-23_09-10_CDT"
TIMEZONE_NAME = "America/Chicago"
RESUME_RUN_ID = None

DEVICE = "cuda"
DTYPE = "auto"
ALL_MODEL_NAMES = ["gpt2-small", "qwen-0.6b"]
EVALUATION_MODEL_NAMES = ["gpt2-small", "qwen-0.6b"]
REUSE_COMPLETED_MODEL_NAMES = []
MODEL_BATCH_SIZES = {"gpt2-small": 16, "qwen-0.6b": 8}
RUN_EVALUATION = True

## 1. Install and verify the project

The notebook uses the reusable AIT transfer configuration and frozen-direction evaluation APIs. Dataset loading, artifact validation, patching, metrics, and persistence remain in `sentiment_geometry`.

In [ ]:
import importlib
import os
import pkgutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
for loaded_name in list(sys.modules):
    if loaded_name == "sentiment_geometry" or loaded_name.startswith("sentiment_geometry."):
        del sys.modules[loaded_name]
importlib.invalidate_caches()

import sentiment_geometry
expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(sentiment_geometry.__file__).resolve().parent
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )
module_names = sorted(
    module.name
    for module in pkgutil.walk_packages(sentiment_geometry.__path__, prefix="sentiment_geometry.")
    if module.name != "sentiment_geometry.__main__"
)
for module_name in module_names:
    importlib.import_module(module_name)
required_apis = {
    "sentiment_geometry.experiments": {
        "AITValenceTransferConfig",
        "load_frozen_direction_selections",
        "run_frozen_sentiment_direction_evaluation",
    },
    "sentiment_geometry.persistence": {
        "RunArtifactStore", "maybe_mount_google_drive", "prepare_timestamped_run"
    },
}
for module_name, api_names in required_apis.items():
    module = importlib.import_module(module_name)
    missing_apis = sorted(name for name in api_names if not hasattr(module, name))
    if missing_apis:
        raise ImportError(f"{module_name} is missing {missing_apis}.")
print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)

## 2. Mount Drive and configure the locked evaluation

The YAML pins both model revisions, all four directed-pair dataset revisions, the source run, validation-selection provenance, locked AIT-test provenance, and method-specific patch positions. The source run is read-only; outputs go to a separate minute-stamped experiment directory.

In [ ]:
import json

import pandas as pd
import torch
from IPython.display import display

from sentiment_geometry.experiments import AITValenceTransferConfig
from sentiment_geometry.persistence import (
    RunArtifactStore, maybe_mount_google_drive, prepare_timestamped_run
)

if RUN_EVALUATION and DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")
maybe_mount_google_drive(True)
plan = AITValenceTransferConfig.load(PROJECT_ROOT / CONFIG_RELATIVE_PATH)
if [model.name for model in plan.models] != ALL_MODEL_NAMES:
    raise RuntimeError("The pinned model list changed unexpectedly.")
if set(MODEL_BATCH_SIZES) != set(EVALUATION_MODEL_NAMES):
    raise RuntimeError("MODEL_BATCH_SIZES must specify only evaluated models.")

SOURCE_RUN_ROOT = plan.source_run_root(
    DRIVE_STORAGE_ROOT, source_run_id=SOURCE_RUN_ID
)
SOURCE_MANIFEST_PATH = SOURCE_RUN_ROOT / "run_manifest.json"
if not SOURCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(SOURCE_MANIFEST_PATH)
source_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text(encoding="utf-8"))
if source_manifest.get("status") != "completed":
    raise RuntimeError(
        f"Source run {SOURCE_RUN_ID} is not completed: {source_manifest.get('status')!r}"
    )

if "RUN_LAYOUT" not in globals() or RESUME_RUN_ID is not None:
    RUN_LAYOUT = prepare_timestamped_run(
        DRIVE_STORAGE_ROOT,
        experiment_name=plan.output_experiment_name,
        timezone_name=TIMEZONE_NAME,
        resume_run_id=RESUME_RUN_ID,
    )
config = plan.build_evaluation_config(
    storage_root=DRIVE_STORAGE_ROOT,
    source_run_id=SOURCE_RUN_ID,
    output_dir=RUN_LAYOUT.results_dir,
    evaluated_model_names=EVALUATION_MODEL_NAMES,
    reuse_completed_models=REUSE_COMPLETED_MODEL_NAMES,
    device=DEVICE,
    dtype=DTYPE,
    batch_sizes=MODEL_BATCH_SIZES,
)
METHOD_ORDER = list(plan.methods)
DATASET_ORDER = [dataset.name for dataset in plan.datasets]
RunArtifactStore(RUN_LAYOUT.root).write_json("requested_config.json", config.to_dict())
RUN_LAYOUT.update_manifest(
    status="configured",
    metadata={
        "project_commit": project_commit,
        "source_run_id": SOURCE_RUN_ID,
        "selection_dataset": plan.selection_dataset,
        "source_evaluation_dataset": plan.source_evaluation_dataset,
        "evaluated_models": EVALUATION_MODEL_NAMES,
        "reused_completed_models": REUSE_COMPLETED_MODEL_NAMES,
        "evaluation_datasets": DATASET_ORDER,
        "method_patch_positions": plan.method_patch_positions,
        "configuration": "requested_config.json",
    },
)
display(pd.DataFrame([
    {
        "dataset": dataset.name, "repository": dataset.repo_id,
        "revision": dataset.revision, "split": dataset.split
    }
    for dataset in plan.datasets
]))
print("Parent run:  ", SOURCE_RUN_ROOT)
print("New run ID:  ", RUN_LAYOUT.run_id)
print("New run root:", RUN_LAYOUT.root)

## 3. Authenticate to the private Hugging Face datasets

The token is hidden, used only while resolving/loading the private datasets, and then removed from the process and notebook cache.

In [ ]:
import gc
from getpass import getpass

from huggingface_hub import HfApi
from huggingface_hub.utils import reset_sessions

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): " ).strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def clear_hf_credentials(env_name="HF_TOKEN"):
    os.environ.pop(env_name, None)
    value = _RUNTIME_SECRETS.pop("HF_TOKEN", None)
    if value is not None:
        del value
    reset_sessions()
    gc.collect()

_token = get_runtime_secret("HF_TOKEN")
try:
    hf_account = HfApi(token=_token).whoami()["name"]
except BaseException:
    clear_hf_credentials()
    raise
finally:
    del _token
print(f"Authenticated to Hugging Face as {hf_account}. Token value was not displayed.")

## 4. Load the frozen full-AIT best layers

This reads one validation-selected layer and its checkpoint for every model/method from the source run. It also verifies that the source run's locked `ait_test` result used that same layer. Each displayed cell is `Lxx (AIT-validation logit-flip %)`.

In [ ]:
from sentiment_geometry.experiments import load_frozen_direction_selections

MODEL_LABELS = {
    "gpt2-small": "GPT-2 Small",
    "qwen-0.6b": "Qwen3-0.6B Base",
}
METHOD_LABELS = {
    "mean_diff": "Mean Difference",
    "das": "DAS (1D)",
}
selection_rows = []
for model in plan.models:
    for selected in load_frozen_direction_selections(config, model):
        selection_rows.append({
            "model": selected.model, "method": selected.method,
            "fit_position": selected.fit_position,
            "selected_layer": selected.selected_layer,
            "selection_dataset": selected.selection_dataset,
            "selection_metric": selected.selection_metric,
            "selection_value_percent": selected.selection_value_percent,
            "patch_position": plan.method_patch_positions[selected.method],
            "source_direction_checkpoint": str(selected.checkpoint_path),
        })
FROZEN_SELECTION = pd.DataFrame(selection_rows)
if len(FROZEN_SELECTION) != len(ALL_MODEL_NAMES) * len(METHOD_ORDER):
    raise RuntimeError("The frozen full-AIT selection grid is incomplete.")
if set(FROZEN_SELECTION["selection_dataset"]) != {"ait_eval"}:
    raise RuntimeError("A layer was not selected on AIT validation.")
if set(FROZEN_SELECTION["selection_metric"]) != {"logit_flip_percent"}:
    raise RuntimeError("A layer was not selected by validation logit-flip percent.")
layer_display = FROZEN_SELECTION.copy()
layer_display["layer_and_score"] = layer_display.apply(
    lambda row: f"L{int(row['selected_layer']):02d} ({float(row['selection_value_percent']):.1f}%)",
    axis=1,
)
best_layer_table = layer_display.pivot(
    index="method", columns="model", values="layer_and_score"
).reindex(index=METHOD_ORDER, columns=ALL_MODEL_NAMES)
best_layer_table.index = [METHOD_LABELS[method] for method in METHOD_ORDER]
best_layer_table.columns = [MODEL_LABELS[model] for model in ALL_MODEL_NAMES]
display(best_layer_table.style.set_properties(**{"text-align": "center"}))
display(FROZEN_SELECTION[[
    "model", "method", "selected_layer", "patch_position",
    "selection_dataset", "selection_value_percent"
]].sort_values(["model", "method"]).reset_index(drop=True))

## 5. Run the locked transfer evaluations

The package writes aggregate metrics, per-case records, frozen-layer provenance, dataset summaries, direction metadata, and resolved configuration files to Drive.

In [ ]:
from sentiment_geometry.experiments import run_frozen_sentiment_direction_evaluation

if not RUN_EVALUATION:
    print("Evaluation execution is disabled. Set RUN_EVALUATION = True.")
else:
    os.environ.pop(config.hf_token_env, None)
    try:
        os.environ[config.hf_token_env] = get_runtime_secret("HF_TOKEN")
        RUN_LAYOUT.update_manifest(status="running")
        completed_results_dir = run_frozen_sentiment_direction_evaluation(config)
        if completed_results_dir.resolve() != RUN_LAYOUT.results_dir.resolve():
            raise RuntimeError(f"Unexpected results directory: {completed_results_dir}")
        RUN_LAYOUT.update_manifest(status="evaluation-completed")
    except BaseException as error:
        RUN_LAYOUT.update_manifest(
            status="failed", metadata={"failure_type": type(error).__name__}
        )
        raise
    finally:
        clear_hf_credentials(config.hf_token_env)
    print("Evaluation results saved to:", completed_results_dir)

## 6. Audit and display essential results

The audit requires the complete model × method × dataset grid, the frozen validation-selected layers, the configured patch position for every method, pinned/resolved dataset provenance, and all per-case records before completion.

In [ ]:
from datetime import datetime
from itertools import product
from zoneinfo import ZoneInfo

METRICS_PATH = RUN_LAYOUT.results_dir / "all_models_metrics.csv"
SELECTION_PATH = RUN_LAYOUT.results_dir / "all_models_layer_selection.csv"
SUMMARY_PATH = RUN_LAYOUT.results_dir / "all_models_dataset_summary.csv"
for required_path in (METRICS_PATH, SELECTION_PATH, SUMMARY_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)
metrics = pd.read_csv(METRICS_PATH)
saved_selection = pd.read_csv(SELECTION_PATH)
dataset_summary = pd.read_csv(SUMMARY_PATH)
metric_keys = ["model", "method", "fit_position", "dataset"]
if metrics.duplicated(metric_keys).any():
    raise RuntimeError("Duplicate aggregate metric cells were saved.")
expected_cells = set(product(ALL_MODEL_NAMES, METHOD_ORDER, ["final"], DATASET_ORDER))
actual_cells = set(metrics[metric_keys].itertuples(index=False, name=None))
if expected_cells != actual_cells:
    raise RuntimeError(
        f"Incomplete metric grid. Missing={sorted(expected_cells - actual_cells)}; "
        f"unexpected={sorted(actual_cells - expected_cells)}"
    )
for method, position in plan.method_patch_positions.items():
    observed = set(metrics.loc[metrics["method"] == method, "patch_position"])
    if observed != {position}:
        raise RuntimeError(f"Unexpected patch position for {method}: {observed}")
if set(saved_selection["selection_dataset"]) != {"ait_eval"}:
    raise RuntimeError("Saved layers were not selected exclusively on AIT validation.")
for model_name in ALL_MODEL_NAMES:
    model_dir = RUN_LAYOUT.results_dir / model_name
    for filename in (
        "metrics.csv", "patching_records.csv", "layer_selection.csv",
        "dataset_summary.csv", "direction_metadata.csv", "resolved_config.json"
    ):
        if not (model_dir / filename).is_file():
            raise FileNotFoundError(model_dir / filename)
    patching_records = pd.read_csv(model_dir / "patching_records.csv")
    expected_records = int(
        dataset_summary.loc[dataset_summary["model"] == model_name, "n_directed_cases"].sum()
    ) * len(METHOD_ORDER)
    if len(patching_records) != expected_records:
        raise RuntimeError(
            f"{model_name} saved {len(patching_records)} per-case records; "
            f"expected {expected_records}."
        )
completed_at = datetime.now(ZoneInfo(RUN_LAYOUT.timezone_name)).isoformat(timespec="minutes")
RUN_LAYOUT.update_manifest(
    status="completed",
    metadata={
        "completed_at": completed_at, "aggregate_metric_rows": len(metrics),
        "frozen_layer_rows": len(saved_selection),
        "dataset_summary_rows": len(dataset_summary),
    },
)
display(dataset_summary.sort_values(["model", "dataset"]).reset_index(drop=True))
print(f"Validated {len(metrics)} aggregate metric rows and all per-case records.")

In [ ]:
from IPython.display import Markdown

DATASET_LABELS = {
    "sst": "SST", "imdb": "IMDb",
    "dynasent_r1": "DynaSent R1", "dynasent_r2": "DynaSent R2",
}
TABLE_METRICS = {
    "logit_flip_percent": "Logit Flip Percent",
    "sign_flip_percent": "Literal Sign Flip Percent",
}
for model_name in ALL_MODEL_NAMES:
    model_metrics = metrics[metrics["model"] == model_name].copy()
    for metric_column, metric_title in TABLE_METRICS.items():
        if model_metrics[metric_column].isna().any():
            raise RuntimeError(f"Missing {metric_column} values for {model_name}.")
        table = model_metrics.pivot(
            index="method", columns="dataset", values=metric_column
        ).reindex(index=METHOD_ORDER, columns=DATASET_ORDER)
        table.index = [METHOD_LABELS[method] for method in METHOD_ORDER]
        table.columns = [DATASET_LABELS[dataset] for dataset in DATASET_ORDER]
        display(Markdown(f"### {MODEL_LABELS[model_name]} — {metric_title}"))
        display(table.style.format("{:.1f}").set_properties(**{"text-align": "center"}))

## 7. Final paths and credential cleanup

In [ ]:
clear_hf_credentials(config.hf_token_env)
assert config.hf_token_env not in os.environ
assert "HF_TOKEN" not in _RUNTIME_SECRETS
assert "_token" not in globals()
print("Verified: HF_TOKEN is absent from the environment and notebook secret cache.")
print("Completed run:", RUN_LAYOUT.root)
print("Manifest:     ", RUN_LAYOUT.manifest_path)
print("Combined CSVs:", RUN_LAYOUT.results_dir)
for model_name in ALL_MODEL_NAMES:
    print("Per-model:   ", RUN_LAYOUT.results_dir / model_name)

## Interpretation boundary

These are locked cross-dataset transfer results for valence directions learned on full AIT training data and selected on AIT validation. SST, IMDb, DynaSent R1, and DynaSent R2 are confirmation/OOD datasets only: their results must not be used to reselect a layer, DAS checkpoint, method, patch position, direction sign, or intervention strength.